In [8]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem

chembl = pd.read_csv("dataset_chembl/chembl.csv")   # must contain 'smiles'
pubchem = pd.read_csv("lit/eel_standardized_output.csv") # must contain 'smiles'

print(chembl.shape, pubchem.shape)

def standardize_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        return Chem.MolToSmiles(mol, canonical=True)
    except:
        return None

chembl["canonical_smiles"] = chembl["standard_smiles"].apply(standardize_smiles)
pubchem["canonical_smiles"] = pubchem["standard_smiles"].apply(standardize_smiles)

chembl = chembl.dropna(subset=["canonical_smiles"])
pubchem = pubchem.dropna(subset=["canonical_smiles"])

def smiles_to_inchikey(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return Chem.MolToInchiKey(mol)
    except:
        return None

chembl["inchikey"] = chembl["canonical_smiles"].apply(smiles_to_inchikey)
pubchem["inchikey"] = pubchem["canonical_smiles"].apply(smiles_to_inchikey)
                                                        
chembl = chembl.drop_duplicates(subset="inchikey")
pubchem = pubchem.drop_duplicates(subset="inchikey")
                                                        
overlap = set(chembl["inchikey"]) & set(pubchem["inchikey"])
print(f"Number of overlapping compounds: {len(overlap)}")



(4620, 2) (128, 2)
Number of overlapping compounds: 0


In [9]:
pubchem_clean = pubchem[~pubchem["inchikey"].isin(overlap)]

In [10]:
pubchem_clean.to_csv("lit/eel2_clean.csv", index=False)